In [1]:
!pip install ipython-sql
%load_ext sql
%sql sqlite:///my_data_analysis.db

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [3]:
import pandas as pd
import sqlite3
import os

# Connect to the database created above
conn = sqlite3.connect('my_data_analysis.db')

# file mapping
my_files = ['ravenstack_accounts.csv', 'ravenstack_churn_events.csv', 'ravenstack_feature_usage.csv', 'ravenstack_subscriptions.csv', 'ravenstack_support_tickets.csv']

table_mapping = {
    'ravenstack_accounts.csv' : 'accounts',
    'ravenstack_churn_events.csv' : 'churn_events',
    'ravenstack_feature_usage.csv' : 'feature_usage',
    'ravenstack_subscriptions.csv' : 'subscriptions',
    'ravenstack_support_tickets.csv' : 'support_tickets'

}

for file_name, table_name in table_mapping.items():
    df = pd.read_csv(f'/content/{file_name}')
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Loaded '{file_name}' as table: '{table_name}'")


Loaded 'ravenstack_accounts.csv' as table: 'accounts'
Loaded 'ravenstack_churn_events.csv' as table: 'churn_events'
Loaded 'ravenstack_feature_usage.csv' as table: 'feature_usage'
Loaded 'ravenstack_subscriptions.csv' as table: 'subscriptions'
Loaded 'ravenstack_support_tickets.csv' as table: 'support_tickets'


In [4]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [5]:
%%sql
SELECT
	a.account_id,
    a.plan_tier,
    s.subscription_id,
    s.mrr_amount
FROM accounts a
JOIN subscriptions s
ON a.account_id = s.account_id
LIMIT 10;

 * sqlite:///my_data_analysis.db
Done.


account_id,plan_tier,subscription_id,mrr_amount
A-2e4581,Basic,S-1e349b,1938
A-2e4581,Basic,S-34a131,1791
A-2e4581,Basic,S-3d7bed,836
A-2e4581,Basic,S-7ce677,441
A-2e4581,Basic,S-b78829,980
A-2e4581,Basic,S-c4cef7,637
A-2e4581,Basic,S-d24841,0
A-2e4581,Basic,S-d4d459,5771
A-2e4581,Basic,S-e79dbc,0
A-2e4581,Basic,S-faa8ec,209


In [6]:
%%sql
 SELECT
	a.account_id,
	a.plan_tier,
    f.feature_name,
    f.usage_count
FROM accounts a
JOIN subscriptions s
	ON a.account_id = s.account_id
JOIN feature_usage f
	ON s.subscription_id = f.subscription_id
LIMIT 10;

 * sqlite:///my_data_analysis.db
Done.


account_id,plan_tier,feature_name,usage_count
A-2e4581,Basic,feature_13,13
A-2e4581,Basic,feature_15,13
A-2e4581,Basic,feature_25,13
A-2e4581,Basic,feature_34,8
A-2e4581,Basic,feature_34,9
A-2e4581,Basic,feature_4,10
A-2e4581,Basic,feature_13,11
A-2e4581,Basic,feature_2,9
A-2e4581,Basic,feature_28,12
A-2e4581,Basic,feature_39,9


**ACTIVATION**

In [7]:
# What % of users perform their first meaningful action after signup?
%%sql
SELECT
  COUNT(DISTINCT a.account_id) AS total_users,
  COUNT(DISTINCT f.account_id) AS activated_users,
  COUNT(DISTINCT f.account_id) * 100.0 / COUNT (DISTINCT a.account_id) AS activation_rate
  FROM accounts a
  LEFT JOIN (
    SELECT DISTINCT s.account_id
    FROM subscriptions s
    JOIN feature_usage f
      ON s.subscription_id = f.subscription_id
  ) f
  On a.account_id = f.account_id;

 * sqlite:///my_data_analysis.db
Done.


total_users,activated_users,activation_rate
500,500,100.0


In [8]:
# How long does it take users to reach their first meaningful interaction?
%%sql
SELECT
AVG(julianday(f.first_usage_date) - julianday(a.signup_date)) AS avg_days_to_activation
FROM accounts a
JOIN (
  SELECT
    s.account_id,
    MIN(f.usage_date) AS first_usage_date
  FROM subscriptions s
  JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
  GROUP BY s.account_id
  )f
  ON a.account_id = f.account_id
  WHERE julianday(f.first_usage_date) >= julianday(a.signup_date);

 * sqlite:///my_data_analysis.db
Done.


avg_days_to_activation
5.2


In [9]:
# Which early behaviors indicate a user is likely to become active?
%%sql

SELECT
   f.feature_name,
   COUNT(*) AS usage_count
FROM subscriptions s
JOIN feature_usage f
  ON s.subscription_id = f.subscription_id
GROUP BY f.feature_name
ORDER BY usage_count DESC;


 * sqlite:///my_data_analysis.db
Done.


feature_name,usage_count
feature_32,659
feature_12,659
feature_6,655
feature_17,651
feature_34,650
feature_26,649
feature_36,648
feature_31,644
feature_38,643
feature_24,643


***ACTIVATION SUMMARY:***

Measures how quickly users complete their first meaningful action after signup, indicating how fast they reach initial product value and begin interacting with core workflows.

**ENGAGEMENT**

In [10]:
# How frequently do users interact with the product over time?
%%sql
SELECT
  a.account_id,
  COUNT(f.usage_id) AS total_interactions,
  COUNT(DISTINCT f.usage_date) AS active_days
FROM accounts a
JOIN subscriptions s
  ON a.account_id = s.account_id
JOIN feature_usage f
  ON s.subscription_id = f.subscription_id
GROUP BY a.account_id
ORDER BY total_interactions DESC
LIMIT 10;

 * sqlite:///my_data_analysis.db
Done.


account_id,total_interactions,active_days
A-cab532,101,96
A-592832,101,92
A-726cfa,98,88
A-7df7a7,91,86
A-40de06,91,84
A-7f4db3,90,88
A-812c5b,88,84
A-6f8ad2,88,87
A-781cc0,87,81
A-692f18,87,83


In [11]:
# Which features drive the highest engagement?
%%sql
SELECT
  feature_name,
  COUNT(*) AS total_usage,
  AVG(usage_count) AS avg_usage_per_event
FROM feature_usage
GROUP BY feature_name
ORDER BY total_usage DESC
LIMIT 10;

 * sqlite:///my_data_analysis.db
Done.


feature_name,total_usage,avg_usage_per_event
feature_32,659,10.14567526555387
feature_12,659,9.915022761760243
feature_6,655,9.993893129770992
feature_17,651,9.920122887864823
feature_34,650,10.055384615384616
feature_26,649,9.969183359013867
feature_36,648,9.859567901234568
feature_31,644,9.976708074534162
feature_38,643,10.074650077760497
feature_24,643,9.934681181959565


In [12]:
# Do highly engaged users behave differently from low-engagement users?
%%sql
SELECT
  a.account_id,
  COUNT(f.usage_id) AS total_usage,
  a.churn_flag
FROM accounts a
JOIN subscriptions s
  ON a.account_id = s.account_id
JOIN feature_usage f
  ON s.subscription_id = f.subscription_id
GROUP BY a.account_id
LIMIT 10;

 * sqlite:///my_data_analysis.db
Done.


account_id,total_usage,churn_flag
A-00bed1,51,0
A-00cac8,58,0
A-0158bb,36,0
A-016043,47,0
A-019782,55,0
A-029f69,84,0
A-02cd81,72,0
A-02fac6,53,0
A-034368,28,0
A-0354fe,45,0


In [13]:
%%sql

SELECT
    CASE
        WHEN user_usage.total_usage > 60 THEN 'High Engagement'
        ELSE 'Low Engagement'
    END AS engagement_level,
    COUNT(*) AS users,
    AVG(user_usage.total_usage) AS avg_usage
FROM (
    SELECT
        a.account_id,
        COUNT(f.usage_id) AS total_usage
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY a.account_id
) user_usage
GROUP BY engagement_level;

 * sqlite:///my_data_analysis.db
Done.


engagement_level,users,avg_usage
High Engagement,140,70.84285714285714
Low Engagement,360,41.894444444444446


***ENGAGEMENT SUMMARY:***

Captures the depth and frequency of user interactions over time, reflecting how consistently users return and how broadly they use different product features.

**RETENTION**

In [14]:
# % of users returning after 7 days
%%sql
SELECT
COUNT(DISTINCT a.account_id) AS total_users,
COUNT(DISTINCT r.account_id) AS retaned_users_7d,
COUNT(DISTINCT r.account_id) * 100.0 / COUNT(DISTINCT a.account_id) AS retention_7d
FROM accounts a
LEFT JOIN (
  SELECT DISTINCT s.account_id
  FROM subscriptions s
  JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
  JOIN accounts a2
    ON s.account_id = a2.account_id
  WHERE julianday(f.usage_date) - julianday(a2.signup_date) <=7
)r
ON a.account_id = r.account_id;

 * sqlite:///my_data_analysis.db
Done.


total_users,retaned_users_7d,retention_7d
500,497,99.4


In [15]:
# % of users returning after 30 days
%%sql
SELECT
COUNT(DISTINCT a.account_id) AS total_users,
COUNT(DISTINCT r.account_id) AS retaned_users_30d,
COUNT(DISTINCT r.account_id) * 100.0 / COUNT(DISTINCT a.account_id) AS retention_30d
FROM accounts a
LEFT JOIN (
  SELECT DISTINCT s.account_id
  FROM subscriptions s
  JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
  JOIN accounts a2
    ON s.account_id = a2.account_id
  WHERE julianday(f.usage_date) - julianday(a2.signup_date) <=30
)r
ON a.account_id = r.account_id;

 * sqlite:///my_data_analysis.db
Done.


total_users,retaned_users_30d,retention_30d
500,500,100.0


In [16]:
# What early behaviors link to retention?
%%sql
SELECT
  CASE
    WHEN early_usage.usage_count > 10 THEN 'High Early Activity'
    ELSE 'Low Early Activity'
  END AS early_behavior,
  COUNT(*) AS users
  FROM (
  SELECT
    a.account_id,
    COUNT(f.usage_id) AS usage_count
  FROM accounts a
  JOIN subscriptions s
    ON a.account_id = s.account_id
  JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
  WHERE julianday (f.usage_date) - julianday(a.signup_date) <= 7
  GROUP BY a.account_id
  ) early_usage
  GROUP BY  early_behavior;

 * sqlite:///my_data_analysis.db
Done.


early_behavior,users
High Early Activity,406
Low Early Activity,91


***RETENTION SUMMARY:***

Tracks whether users continue returning to the product over time, indicating the formation of consistent usage habits and long-term product dependency.

**CHURN**

In [17]:
#What patterns do churned users show?
%%sql

SELECT
    a.churn_flag,
    COUNT(DISTINCT a.account_id) AS users,
    AVG(f.usage_count) AS avg_usage,
    AVG(f.error_count) AS avg_errors
FROM accounts a
JOIN subscriptions s
    ON a.account_id = s.account_id
JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
GROUP BY a.churn_flag;

 * sqlite:///my_data_analysis.db
Done.


churn_flag,users,avg_usage,avg_errors
0,390,10.018210116731517,0.5712062256809338
1,110,10.030393013100436,0.5409606986899563


In [18]:
#Do low activity users churn more?
%%sql
SELECT
    CASE
        WHEN user_usage.total_usage > 60 THEN 'High Activity'
        ELSE 'Low Activity'
    END AS activity_level,
    a.churn_flag,
    COUNT(*) AS users
FROM (
    SELECT
        a.account_id,
        COUNT(f.usage_id) AS total_usage
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY a.account_id
) user_usage
JOIN accounts a
    ON user_usage.account_id = a.account_id
GROUP BY activity_level, a.churn_flag;

 * sqlite:///my_data_analysis.db
Done.


activity_level,churn_flag,users
High Activity,0,109
High Activity,1,31
Low Activity,0,281
Low Activity,1,79


In [19]:
#Do users with more errors churn more?
%%sql
SELECT
  a.churn_flag,
  AVG(f.error_count) AS avg_errors
  FROM accounts a
  JOIN subscriptions s
    ON a.account_id = s.account_id
  JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
  GROUP BY a.churn_flag;

 * sqlite:///my_data_analysis.db
Done.


churn_flag,avg_errors
0,0.5712062256809338
1,0.5409606986899563


In [20]:
%%sql

SELECT
    a.churn_flag,
    AVG(user_usage.total_usage) AS avg_total_usage
FROM (
    SELECT
        a.account_id,
        COUNT(f.usage_id) AS total_usage
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY a.account_id
) user_usage
JOIN accounts a
    ON user_usage.account_id = a.account_id
GROUP BY a.churn_flag;

 * sqlite:///my_data_analysis.db
Done.


churn_flag,avg_total_usage
0,49.42307692307692
1,52.04545454545455


In [21]:
%%sql

SELECT
    f.feature_name,
    a.churn_flag,
    COUNT(*) AS usage_count
FROM feature_usage f
JOIN subscriptions s
    ON f.subscription_id = s.subscription_id
JOIN accounts a
    ON s.account_id = a.account_id
GROUP BY f.feature_name, a.churn_flag
ORDER BY f.feature_name;

 * sqlite:///my_data_analysis.db
Done.


feature_name,churn_flag,usage_count
feature_1,0,462
feature_1,1,167
feature_10,0,481
feature_10,1,153
feature_11,0,479
feature_11,1,164
feature_12,0,529
feature_12,1,130
feature_13,0,474
feature_13,1,138


In [22]:
%%sql

SELECT
    a.churn_flag,
    AVG(CASE
        WHEN julianday(f.usage_date) - julianday(a.signup_date) <= 7
        THEN 1 ELSE 0 END) AS early_usage_ratio
FROM accounts a
JOIN subscriptions s
    ON a.account_id = s.account_id
JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
GROUP BY a.churn_flag;

 * sqlite:///my_data_analysis.db
Done.


churn_flag,early_usage_ratio
0,0.5516472114137484
1,0.49519650655021835


***CHURN SUMMARY:***

Identifies users who stop using the product, focusing on behavioral signals leading up to exit and whether disengagement is gradual or still active at the point of churn.

**MONETIZATION**

In [23]:
%%sql

WITH user_summary AS (
    SELECT
        a.account_id,
        s.is_trial,
        s.mrr_amount,
        COUNT(f.usage_id) AS total_usage,
        AVG(f.error_count) AS avg_errors
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY a.account_id
)

SELECT * FROM user_summary LIMIT 5;

 * sqlite:///my_data_analysis.db
Done.


account_id,is_trial,mrr_amount,total_usage,avg_errors
A-00bed1,0,5572,51,0.5294117647058824
A-00cac8,0,7761,58,0.5344827586206896
A-0158bb,1,0,36,0.6111111111111112
A-016043,1,0,47,0.44680851063829785
A-019782,1,0,55,0.5454545454545454


In [24]:
#Free vs Paid users behavior
%%sql

WITH user_summary AS (
    SELECT
        a.account_id,
        s.is_trial,
        s.mrr_amount,
        COUNT(f.usage_id) AS total_usage,
        AVG(f.error_count) AS avg_errors
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY a.account_id
)

SELECT
    is_trial,
    COUNT(*) AS users,
    AVG(total_usage) AS avg_usage,
    AVG(avg_errors) AS avg_errors,
    AVG(mrr_amount) AS avg_mrr
FROM user_summary
GROUP BY is_trial;

 * sqlite:///my_data_analysis.db
Done.


is_trial,users,avg_usage,avg_errors,avg_mrr
0,431,50.20417633410673,0.5639994923247865,2536.5754060324825
1,69,48.72463768115942,0.5641065224651757,0.0


In [25]:
#What actions increase conversion?
%%sql

WITH latest_subscription AS (
    SELECT
        account_id,
        is_trial,
        ROW_NUMBER() OVER (
            PARTITION BY account_id
            ORDER BY start_date DESC
        ) AS rn
    FROM subscriptions
),

final_users AS (
    SELECT
        account_id,
        is_trial
    FROM latest_subscription
    WHERE rn = 1
),

early_usage AS (
    SELECT
        a.account_id,
        COUNT(f.usage_id) AS early_usage
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    WHERE julianday(f.usage_date) - julianday(a.signup_date) <= 7
    GROUP BY a.account_id
)

SELECT
    CASE
        WHEN e.early_usage > 10 THEN 'High Early Activity'
        ELSE 'Low Early Activity'
    END AS early_behavior,
    f.is_trial,
    COUNT(*) AS users
FROM early_usage e
JOIN final_users f
    ON e.account_id = f.account_id
GROUP BY early_behavior, f.is_trial;

 * sqlite:///my_data_analysis.db
Done.


early_behavior,is_trial,users
High Early Activity,0,344
High Early Activity,1,62
Low Early Activity,0,77
Low Early Activity,1,14


In [26]:
# Do highly engaged users generate more revenue?
%%sql

WITH user_summary AS (
    SELECT
        a.account_id,
        s.mrr_amount,
        COUNT(f.usage_id) AS total_usage
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY a.account_id
)

SELECT
    CASE
        WHEN total_usage > 60 THEN 'High Engagement'
        ELSE 'Low Engagement'
    END AS engagement_level,
    COUNT(*) AS users,
    AVG(mrr_amount) AS avg_mrr
FROM user_summary
GROUP BY engagement_level;

 * sqlite:///my_data_analysis.db
Done.


engagement_level,users,avg_mrr
High Engagement,140,2510.1071428571427
Low Engagement,360,2060.6916666666666


***MONETIZATION SUMMARY:***

Measures how user behavior translates into revenue outcomes, particularly how usage patterns relate to conversion, upgrades, and long-term revenue contribution.

In [27]:
%%sql

WITH user_status AS (
    SELECT
        account_id,
        churn_flag
    FROM accounts
),

feature_usage_summary AS (
    SELECT
        a.account_id,
        f.feature_name,
        COUNT(*) AS usage_count
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY a.account_id, f.feature_name
)

SELECT
    u.churn_flag,
    f.feature_name,
    SUM(f.usage_count) AS total_usage
FROM feature_usage_summary f
JOIN user_status u
    ON f.account_id = u.account_id
GROUP BY u.churn_flag, f.feature_name
ORDER BY u.churn_flag, total_usage DESC;

 * sqlite:///my_data_analysis.db
Done.


churn_flag,feature_name,total_usage
0,feature_12,529
0,feature_32,519
0,feature_6,518
0,feature_36,508
0,feature_17,508
0,feature_24,502
0,feature_31,501
0,feature_30,499
0,feature_2,499
0,feature_29,496


In [28]:
# pre activation errors
%%sql

WITH first_usage AS (
    SELECT
        s.account_id,
        MIN(f.usage_date) AS first_usage_date
    FROM subscriptions s
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY s.account_id
),

pre_activation_errors AS (
    SELECT
        a.account_id,
        a.churn_flag,
        AVG(f.error_count) AS avg_errors_before_activation
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    JOIN first_usage fu
        ON a.account_id = fu.account_id
    WHERE f.usage_date <= fu.first_usage_date
    GROUP BY a.account_id
)

SELECT
    churn_flag,
    AVG(avg_errors_before_activation) AS avg_errors
FROM pre_activation_errors
GROUP BY churn_flag;

 * sqlite:///my_data_analysis.db
Done.


churn_flag,avg_errors
0,0.6166666666666667
1,0.40454545454545454


In [29]:
#Consistent vs Inactive Users
%%sql

SELECT
    a.account_id,
    COUNT(*) AS total_usage,
    COUNT(DISTINCT f.usage_date) AS active_days
FROM accounts a
JOIN subscriptions s
    ON a.account_id = s.account_id
JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
GROUP BY a.account_id;

 * sqlite:///my_data_analysis.db
Done.


account_id,total_usage,active_days
A-00bed1,51,49
A-00cac8,58,57
A-0158bb,36,35
A-016043,47,46
A-019782,55,55
A-029f69,84,80
A-02cd81,72,68
A-02fac6,53,51
A-034368,28,27
A-0354fe,45,44


In [30]:
#Frequency Impact on Retention
%%sql

SELECT
    a.churn_flag,
    COUNT(DISTINCT f.usage_date) AS active_days,
    COUNT(*) AS total_usage
FROM accounts a
JOIN subscriptions s
    ON a.account_id = s.account_id
JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
GROUP BY a.churn_flag;

 * sqlite:///my_data_analysis.db
Done.


churn_flag,active_days,total_usage
0,731,19275
1,730,5725


In [31]:
#Steady Engagement vs Churn
%%sql

WITH user_activity AS (
    SELECT
        a.account_id,
        a.churn_flag,
        COUNT(*) AS total_usage,
        COUNT(DISTINCT f.usage_date) AS active_days
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY a.account_id
)

SELECT
    churn_flag,
    CASE
        WHEN active_days >= 5 THEN 'Steady Users'
        ELSE 'Unsteady Users'
    END AS engagement_type,
    COUNT(*) AS users
FROM user_activity
GROUP BY churn_flag, engagement_type;

 * sqlite:///my_data_analysis.db
Done.


churn_flag,engagement_type,users
0,Steady Users,390
1,Steady Users,110


In [32]:
#Growth-Oriented Users (Feature Breadth)
%%sql

SELECT
    s.account_id,
    COUNT(DISTINCT f.feature_name) AS feature_breadth,
    COUNT(*) AS total_usage
FROM subscriptions s
JOIN feature_usage f
    ON s.subscription_id = f.subscription_id
GROUP BY s.account_id;

 * sqlite:///my_data_analysis.db
Done.


account_id,feature_breadth,total_usage
A-00bed1,32,51
A-00cac8,30,58
A-0158bb,19,36
A-016043,26,47
A-019782,28,55
A-029f69,37,84
A-02cd81,31,72
A-02fac6,32,53
A-034368,20,28
A-0354fe,28,45


In [33]:
#Engagement Pattern vs Churn
%%sql

WITH first_usage AS (
    SELECT
        s.account_id,
        MIN(f.usage_date) AS first_usage_date
    FROM subscriptions s
    JOIN feature_usage f
        ON s.subscription_id = f.subscription_id
    GROUP BY s.account_id
),

usage_classified AS (
    SELECT
        s.account_id,
        a.churn_flag,
        CASE
            WHEN f.usage_date <= date(fu.first_usage_date, '+7 day')
            THEN 'Early'
            ELSE 'Late'
        END AS period,
        COUNT(*) AS usage_count
    FROM feature_usage f
    JOIN subscriptions s
        ON f.subscription_id = s.subscription_id
    JOIN accounts a
        ON a.account_id = s.account_id
    JOIN first_usage fu
        ON fu.account_id = s.account_id
    GROUP BY s.account_id, a.churn_flag, period
)

SELECT
    churn_flag,
    period,
    AVG(usage_count) AS avg_usage
FROM usage_classified
GROUP BY churn_flag, period;

 * sqlite:///my_data_analysis.db
Done.


churn_flag,period,avg_usage
0,Early,1.535897435897436
0,Late,47.88717948717949
1,Early,1.4454545454545455
1,Late,50.6


In [34]:
%%sql

WITH first_activity AS (
    SELECT
        a.account_id,
        MIN(DATE(f.event_time)) AS first_date
    FROM feature_usage f
    JOIN accounts a
        ON f.user_id = a.user_id
    GROUP BY a.account_id
),

early_usage AS (
    SELECT
        a.account_id,
        COUNT(*) AS usage_7d,
        COUNT(DISTINCT f.feature_name) AS features_7d
    FROM feature_usage f
    JOIN accounts a
        ON f.user_id = a.user_id
    JOIN first_activity fa
        ON a.account_id = fa.account_id
    WHERE DATE(f.event_time) BETWEEN fa.first_date
                                 AND DATE(fa.first_date, '+7 days')
    GROUP BY a.account_id
)

SELECT
    s.is_trial,
    AVG(e.usage_7d) AS avg_usage_7d,
    AVG(e.features_7d) AS avg_features_7d
FROM early_usage e
JOIN subscriptions s
    ON e.account_id = s.account_id
GROUP BY s.is_trial;

 * sqlite:///my_data_analysis.db
(sqlite3.OperationalError) no such column: f.event_time
[SQL: WITH first_activity AS (
    SELECT 
        a.account_id,
        MIN(DATE(f.event_time)) AS first_date
    FROM feature_usage f
    JOIN accounts a 
        ON f.user_id = a.user_id
    GROUP BY a.account_id
),

early_usage AS (
    SELECT 
        a.account_id,
        COUNT(*) AS usage_7d,
        COUNT(DISTINCT f.feature_name) AS features_7d
    FROM feature_usage f
    JOIN accounts a 
        ON f.user_id = a.user_id
    JOIN first_activity fa 
        ON a.account_id = fa.account_id
    WHERE DATE(f.event_time) BETWEEN fa.first_date 
                                 AND DATE(fa.first_date, '+7 days')
    GROUP BY a.account_id
)

SELECT 
    s.is_trial,
    AVG(e.usage_7d) AS avg_usage_7d,
    AVG(e.features_7d) AS avg_features_7d
FROM early_usage e
JOIN subscriptions s 
    ON e.account_id = s.account_id
GROUP BY s.is_trial;]
(Background on this error at: https://sqlalche.me/e/20/e3q8

In [35]:
#Pre-Conversion Behavior
%%sql

WITH conversions AS (
    SELECT
        subscription_id,
        account_id,
        start_date AS conversion_date
    FROM subscriptions
    WHERE upgrade_flag = 1
),

pre_conversion_usage AS (
    SELECT
        c.account_id,
        SUM(f.usage_count) AS usage_last_7_days,
        COUNT(DISTINCT f.feature_name) AS feature_breadth_last_7_days
    FROM feature_usage f
    JOIN conversions c
        ON f.subscription_id = c.subscription_id
    WHERE DATE(f.usage_date) BETWEEN DATE(c.conversion_date, '-7 days')
                                 AND DATE(c.conversion_date)
    GROUP BY c.account_id
)

SELECT
    AVG(usage_last_7_days) AS avg_usage_before_conversion,
    AVG(feature_breadth_last_7_days) AS avg_feature_breadth_before_conversion
FROM pre_conversion_usage;

 * sqlite:///my_data_analysis.db
Done.


avg_usage_before_conversion,avg_feature_breadth_before_conversion
9.8125,1.03125


In [36]:
#Activity Trend Before Churn
%%sql

WITH churned_subs AS (
    SELECT
        subscription_id,
        account_id,
        end_date AS churn_date
    FROM subscriptions
    WHERE churn_flag = 1
),

activity_split AS (
    SELECT
        c.account_id,
        CASE
            WHEN DATE(f.usage_date) BETWEEN DATE(c.churn_date, '-7 days')
                                        AND DATE(c.churn_date)
                THEN 'last_7_days'
            WHEN DATE(f.usage_date) BETWEEN DATE(c.churn_date, '-14 days')
                                        AND DATE(c.churn_date, '-8 days')
                THEN 'prev_7_days'
        END AS period,
        SUM(f.usage_count) AS usage
    FROM feature_usage f
    JOIN churned_subs c
        ON f.subscription_id = c.subscription_id
    WHERE DATE(f.usage_date) >= DATE(c.churn_date, '-14 days')
    GROUP BY c.account_id, period
)

SELECT
    period,
    AVG(usage) AS avg_usage
FROM activity_split
WHERE period IS NOT NULL
GROUP BY period;

 * sqlite:///my_data_analysis.db
Done.


period,avg_usage
last_7_days,11.96
prev_7_days,10.5
